### Background computation
We will use the nuflux package to compute the backgorund emission probability:
$$
n_b(E_i) = \phi_{\nu+\mu}^\text{atm}(E_i)\times A_{eff}\times \Delta T_{obs}
$$

nuflux uses `nu_cos_zenith` as the input angle for the flux. Since IceCube is in the south pole:
$$
\theta_\text{zenith} = 0 \degree \rightarrow \delta = -90\degree
$$
Therefore:
$$
\theta_\text{zenith}(\delta) = \delta + 90 
$$


We will save this ina csv file so it can be read in easily in AMPEL without having to run nuflux

Possible improvement: We can fit a single powerlaw to all declination bins and save the gamma and flux normalization in the csv, saving computing time

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import astropy.units as u
import pandas as pd
from scipy.interpolate import interp1d


import nuflux

import sys
sys.path.insert(1, '/Users/cristozilleruelo/Rubicube-Ampel-HU-astro/Rubicube/')
from params import p

In [2]:
def zenith_angle(declination):

    '''Converts declination into zenith angle for IceCube detector '''
    if  not isinstance(declination,  u.Quantity):
        declination = declination * u.deg

    return declination.to(u.deg) + 90 * u.deg 

def cos_zenith(declination):
    
    return np.cos(zenith_angle(declination).to(u.rad))


In [3]:
flux = nuflux.makeFlux('honda2006')#('CORSIKA_GaisserH3a_average')
nu_type=nuflux.NuMu

We want to save the atmosphere model values in a csv with columns energy | flux per declination angle in intervals of 5 deg

E_nu (GeV) | -90 - -60 | -60 - -30| 

In [4]:
180/5

36.0

The flux gets saved in units of $\text{GeV}^{-1}\,\text{cm}^{-2}\,\text{s}^{-1}$ as stated in the [documentation](https://docs.icecube.aq/nuflux/main/pythonapi.html#nuflux.FluxFunction.getFlux)

In [5]:
path_bkg_csv =  p['neutrino_bkg_model_path']

declination_bins = []


with open(path_bkg_csv, 'w') as file:

    for dec in range(-90, 90, 10):
        declination_bins.append(f'{dec} - {dec+10}')
    declination_bins.insert(0, 'E_nu (GeV)')

    #write header
    file.write(','.join(declination_bins) + '\n')

    E_range = np.logspace(3, 8, 1000) * u.GeV
    for E_nu in E_range:
        flux_bins = []
        for dec in range(-90, 90, 10):
            bin_decs = []
            for dec_i in range(dec, dec+11):
                nu_cos_zenith = cos_zenith(dec_i * u.deg)
                f = flux.getFlux(nu_type, E_nu, nu_cos_zenith)
                bin_decs.append(f)
            avg_bin_flux = np.mean(np.array(bin_decs))
            flux_bins.append(str(avg_bin_flux))
        flux_bins.insert(0, str(E_nu.value))
        file.write(','.join(flux_bins) + '\n')

#     header = ','.join('E_nu (GeV)', '')

Now we try to read in the file as a pd data frame

In [6]:
df = pd.read_csv(path_bkg_csv)
df

,E_nu (GeV),-90 - -80,-80 - -70,-70 - -60,-60 - -50,-50 - -40,-40 - -30,-30 - -20,-20 - -10,-10 - 0,0 - 10,10 - 20,20 - 30,30 - 40,40 - 50,50 - 60,60 - 70,70 - 80,80 - 90
0,1.000000e+03,1.477397e-11,1.510677e-11,1.580721e-11,1.694410e-11,1.852789e-11,2.086447e-11,2.454954e-11,3.098380e-11,4.187136e-11,4.187136e-11,3.098380e-11,2.454954e-11,2.086447e-11,1.852789e-11,1.694410e-11,1.580721e-11,1.510677e-11,1.477397e-11
1,1.011591e+03,1.421067e-11,1.453165e-11,1.520729e-11,1.630419e-11,1.783303e-11,2.008929e-11,2.364836e-11,2.986251e-11,4.037821e-11,4.037821e-11,2.986251e-11,2.364836e-11,2.008929e-11,1.783303e-11,1.630419e-11,1.520729e-11,1.453165e-11,1.421067e-11
2,1.023317e+03,1.366854e-11,1.397812e-11,1.462983e-11,1.568810e-11,1.716387e-11,1.934254e-11,2.277991e-11,2.878148e-11,3.893797e-11,3.893797e-11,2.878148e-11,2.277991e-11,1.934254e-11,1.716387e-11,1.568810e-11,1.462983e-11,1.397812e-11,1.366854e-11
3,1.035178e+03,1.314680e-11,1.344537e-11,1.407398e-11,1.509497e-11,1.651947e-11,1.862320e-11,2.194299e-11,2.773927e-11,3.754876e-11,3.754876e-11,2.773927e-11,2.194299e-11,1.862320e-11,1.651947e-11,1.509497e-11,1.407398e-11,1.344537e-11,1.314680e-11
4,1.047177e+03,1.264470e-11,1.293263e-11,1.353895e-11,1.452394e-11,1.589893e-11,1.793025e-11,2.113647e-11,2.673450e-11,3.620880e-11,3.620880e-11,2.673450e-11,2.113647e-11,1.793025e-11,1.589893e-11,1.452394e-11,1.353895e-11,1.293263e-11,1.264470e-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,9.549486e+07,8.635585e-30,8.921597e-30,9.540973e-30,1.060203e-29,1.225265e-29,1.501610e-29,2.020259e-29,3.208678e-29,6.221710e-29,6.221710e-29,3.208678e-29,2.020259e-29,1.501610e-29,1.225265e-29,1.060203e-29,9.540973e-30,8.921597e-30,8.635585e-30
996,9.660175e+07,8.275101e-30,8.549173e-30,9.142694e-30,1.015946e-29,1.174118e-29,1.438927e-29,1.935925e-29,3.074735e-29,5.961994e-29,5.961994e-29,3.074735e-29,1.935925e-29,1.438927e-29,1.174118e-29,1.015946e-29,9.142694e-30,8.549173e-30,8.275101e-30
997,9.772147e+07,7.929664e-30,8.192295e-30,8.761041e-30,9.735359e-30,1.125105e-29,1.378860e-29,1.855112e-29,2.946384e-29,5.713119e-29,5.713119e-29,2.946384e-29,1.855112e-29,1.378860e-29,1.125105e-29,9.735359e-30,8.761041e-30,8.192295e-30,7.929664e-30
998,9.885417e+07,7.598648e-30,7.850316e-30,8.395319e-30,9.328965e-30,1.078139e-29,1.321301e-29,1.777672e-29,2.823390e-29,5.474633e-29,5.474633e-29,2.823390e-29,1.777672e-29,1.321301e-29,1.078139e-29,9.328965e-30,8.395319e-30,7.850316e-30,7.598648e-30


In [7]:
# def read_nu_atm(E_nu, dec, p=p):
#     '''
#     Reads from the csv file the nuflux values as a dataframe and 
#     interpolates for the flux value at that energy
#     '''

#     path_csv = p['neutrino_bkg_model_path']
#     with open(path_csv, 'r') as file:
#         nu_model = pd.read_csv(file)
#     #first find declination bin
#     print(pd.header)


#     pass

In [10]:
dec = 10*u.deg
E_nu = 1e3 * u.GeV

def get_nu_bkg_flux(E_nu, dec, p=p):

    #check and set the correct units
    if not isinstance(E_nu, u.quantity.Quantity):
        E_nu = E_nu * u.GeV
    if not isinstance(dec, u.quantity.Quantity):
        dec = dec * u.degree

    path_csv = p['neutrino_bkg_model_path']
    with open(path_csv, 'r') as file:
        nu_model = pd.read_csv(file)

    #first find declination bin
    dec_bins = nu_model.columns[1:]
    for dec_bin in dec_bins:
        min_dec, max_dec = dec_bin.split(' - ')
        if float(min_dec) <= dec.value <= float(max_dec):
            correct_bin = dec_bin #+1 because first column are energy values
            break
    nu_model[[nu_model.columns[0], correct_bin]]

    nu_model[nu_model.columns[0]].values

    #now we interpolate at the correct energy
    f = interp1d(nu_model[nu_model.columns[0]].values, 
                nu_model[correct_bin].values)

    bkg_flux = f(E_nu.to(u.GeV).value)

    #flux units in GeV^-1 cm^-2 s^-1 as stated in: https://docs.icecube.aq/nuflux/main/pythonapi.html#nuflux.FluxFunction.getFlux

    return bkg_flux * u.GeV ** -1 * u.cm ** -2 * u.s ** -1


In [11]:
get_nu_bkg_flux(1e8, 0)

<Quantity 5.2461016e-29 1 / (GeV s cm2)>